In [0]:
# Databricks notebook source

from __future__ import annotations

import uuid

# ============================================================
# Widgets
# ============================================================
dbutils.widgets.text("catalog", "phc_txdot")
catalog = dbutils.widgets.get("catalog").strip()

# ============================================================
# Constants
# ============================================================
SCRIPT_NAME = "459_build_gold_vw_fact_project_item_estimate.py"
RUN_ID = str(uuid.uuid4())

SOURCE_TABLE = f"{catalog}.gold.fact_project_item_estimate"
TARGET_VIEW = f"{catalog}.gold.vw_fact_project_item_estimate"
RUN_LOG_TABLE = f"{catalog}.staging.pipeline_run_log"

# ============================================================
# Helpers
# ============================================================
def sql_escape(value: str) -> str:
    return value.replace("'", "''")


def log_run(status: str, row_count: int | None, message: str) -> None:
    row_count_sql = "NULL" if row_count is None else str(row_count)
    spark.sql(f"""
        INSERT INTO {RUN_LOG_TABLE}
        VALUES (
              '{RUN_ID}'
            , '{SCRIPT_NAME}'
            , '{sql_escape(status)}'
            , {row_count_sql}
            , '{sql_escape(message)}'
            , current_timestamp()
        )
    """)

# ============================================================
# Build gold.vw_fact_project_item_estimate
# ============================================================
try:
    spark.sql(f"DROP VIEW IF EXISTS {TARGET_VIEW}")

    spark.sql(f"""
    CREATE VIEW {TARGET_VIEW} AS
    SELECT
          estimate_item_fact_key                        AS `Estimate Item Fact Key`
        , project_key                                   AS `Project Key`
        , item_specification_key                        AS `Item Specification Key`

        , project_classification_key                    AS `Project Classification Key`
        , county_key                                    AS `County Key`

        , CASE
              WHEN specification_code IS NOT NULL
               AND specification_description IS NOT NULL
              THEN concat(CAST(specification_code AS STRING), ' - ', specification_description)
              WHEN specification_code IS NOT NULL
              THEN CAST(specification_code AS STRING)
              ELSE specification_description
          END                                           AS `Specification`

        , md5(COALESCE(item_work_category, ''))         AS `Item Work Category Key`

        , project_id                                    AS `Project ID`
        , project_name                                  AS `Project Name`
        , project_number                                AS `Project Number`
        , state_project_number                          AS `State Project Number`
        , control_section_job_csj                       AS `Control Section Job CSJ`
        , controlling_project_id_ccsj                   AS `Controlling Project ID CCSJ`
        , project_type                                  AS `Project Type`
        , project_classification                        AS `Project Classification`
        , CAST(project_actual_let_date AS DATE)         AS `Project Actual Let Date`
        , project_estimated_let_date                    AS `Project Estimated Let Date`
        , county                                        AS `County`
        , location                                      AS `Location`
        , district_division                             AS `District Division`
        , highway                                       AS `Highway`
        , short_description                             AS `Short Description`

        , bid_code                                      AS `Bid Code`
        , alternative_bid_code                          AS `Alternative Bid Code`
        , bid_item_sequence_number                      AS `Bid Item Sequence Number`
        , bid_item_description                          AS `Bid Item Description`
        , measurement_unit                              AS `Measurement Unit`
        , bid_item_quantity                             AS `Bid Item Quantity`
        , specification_code                            AS `Specification Code`
        , specification_description                     AS `Specification Description`
        , effective_specification_description           AS `Effective Specification Description`
        , item_work_category                            AS `Item Work Category`
        , item_work_category_source                     AS `Item Work Category Source`
        , is_defaulted_work_category                    AS `Is Defaulted Work Category`
        , engineer_estimate_unit_price                  AS `Engineer Estimate Unit Price`
        , extended_estimate_item_amount_calc            AS `Extended Estimate Item Amount Calc`
        , project_engineer_total                        AS `Project Engineer Total`
        , engineer_project_total_source                 AS `Engineer Project Total Source`
        , estimate_item_amount_as_pct_of_project_total  AS `Estimate Item Amount As Pct Of Project Total`
        , estimate_item_amount_as_pct_of_item_rollup_diagnostic AS `Estimate Item Amount As Pct Of Item Rollup Diagnostic`
        , estimate_item_rank                            AS `Estimate Item Rank`
        , estimate_item_version_count                   AS `Estimate Item Version Count`
        , estimate_item_has_multiple_versions           AS `Estimate Item Has Multiple Versions`
        , estimate_item_changed_across_versions         AS `Estimate Item Changed Across Versions`
        , is_nonstandard_estimate_item                  AS `Is Nonstandard Estimate Item`
        , is_standard_comparison_item                   AS `Is Standard Comparison Item`
        , is_payment_adjustment_item                    AS `Is Payment Adjustment Item`
        , is_special_deduction_item                     AS `Is Special Deduction Item`
        , is_nonstandard_adjustment_item                AS `Is Nonstandard Adjustment Item`
    FROM {SOURCE_TABLE}
    """)

    row_count = spark.sql(f"SELECT COUNT(*) AS row_count FROM {TARGET_VIEW}").collect()[0]["row_count"]

    log_run("SUCCESS", row_count, "Built gold.vw_fact_project_item_estimate successfully.")

    print("=" * 70)
    print("Built gold.vw_fact_project_item_estimate")
    print(f"Row count: {row_count:,}")
    print("=" * 70)

except Exception as exc:
    log_run("FAILED", None, str(exc))
    raise